In [1]:
!pip install uv
!uv pip install lightning
!uv pip install pytorch-metric-learning
!uv pip install split-folders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 79.1 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.12 environment at: /usr
Resolved 42 packages in 330ms                                        
Prepared 1 package in 96ms                                               
Installed 1 package in 9ms                                  
 + lightning==2.6.0
Using Python 3.12.12 environment at: /usr
Resolved 32 packages in 173ms                                        
Prepared 1 package in 41ms                                               
Installed 1 package in 3msning==2.9.0                       
 + pytorch-metric-learning==2.9.0
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 495ms                                          
Prepared 1 package in 17ms                                               
Installed 1 package in 3ms1                                 
 + split-folders==0.5.1


In [2]:
import os
import random
import splitfolders
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.functional as F
import torchvision.transforms as T
import lightning as pl

from lightning.pytorch import Trainer
from transformers import AutoModel, AutoProcessor
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder

2026-01-26 17:47:30.100625: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769449650.293784      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769449650.354786      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769449650.822745      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769449650.822780      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769449650.822783      55 computation_placer.cc:177] computation placer alr

In [3]:
splitfolders.ratio("/kaggle/input/waste-classification-data/DATASET/TRAIN", output="output",
    seed=1337, ratio=(.8, .2), group_prefix=None, move=False) # default values

Copying files: 22564 files [03:19, 113.09 files/s]


In [4]:
class CFG:
    seed = 42
    model_checkpoint = '/kaggle/input/dinov2/pytorch/base/1'
    img_size = 224
    num_epoch = 8
    
    ## DATA PATHS ##
    # ORIGINAL PATHS
    ROOT_FOLDER_PATH = '/kaggle/input/waste-classification-data/DATASET'
    TRAIN_FOLDER_PATH = os.path.join(ROOT_FOLDER_PATH, 'TRAIN')
    TEST_FOLDER_PATH = os.path.join(ROOT_FOLDER_PATH, 'TEST')

    # SPLIT PATHS (TRAIN/VAL)
    SPLIT_ROOT_FOLDER_PATHS = '/kaggle/working/output'
    SPLIT_TRAIN_FOLDER_PATH = os.path.join(SPLIT_ROOT_FOLDER_PATHS, 'train')
    SPLIT_VAL_FOLDER_PATH = os.path.join(SPLIT_ROOT_FOLDER_PATHS, 'val')

In [5]:
pl.seed_everything(CFG.seed, workers=True)

Seed set to 42


42

In [6]:
print(CFG.ROOT_FOLDER_PATH)
print(CFG.TRAIN_FOLDER_PATH)
print(CFG.TEST_FOLDER_PATH)

/kaggle/input/waste-classification-data/DATASET
/kaggle/input/waste-classification-data/DATASET/TRAIN
/kaggle/input/waste-classification-data/DATASET/TEST


In [7]:
processor = AutoProcessor.from_pretrained(CFG.model_checkpoint, use_fast=True)
processor

BitImageProcessorFast {
  "crop_size": {
    "height": 224,
    "width": 224
  },
  "data_format": "channels_first",
  "default_to_square": false,
  "device": null,
  "disable_grouping": null,
  "do_center_crop": true,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_pad": null,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.485,
    0.456,
    0.406
  ],
  "image_processor_type": "BitImageProcessorFast",
  "image_std": [
    0.229,
    0.224,
    0.225
  ],
  "input_data_format": null,
  "pad_size": null,
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "return_tensors": null,
  "size": {
    "shortest_edge": 256
  }
}

In [8]:
train_augmenter = T.Compose([
    T.Resize(CFG.img_size),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=45),
])

In [9]:
def train_transform(img):
    img = train_augmenter(img)
    inputs = processor(img, return_tensors='pt')
    return inputs['pixel_values'].squeeze(0)

def val_transform(img):
    inputs = processor(img, return_tensors='pt')
    return inputs['pixel_values'].squeeze(0)

In [10]:
CFG.SPLIT_TRAIN_FOLDER_PATH

'/kaggle/working/output/train'

In [11]:
# create dataset
train_dataset = ImageFolder(root=CFG.SPLIT_TRAIN_FOLDER_PATH)
val_dataset = ImageFolder(root=CFG.SPLIT_VAL_FOLDER_PATH)

# apply image transformations
train_dataset.transform = train_transform
val_dataset.transform = val_transform

# create dataloader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# get the label's num_classes
num_classes = len(train_dataset.classes)
num_classes

2

In [12]:
len(train_dataset), len(val_dataset)

(18051, 4513)

In [13]:
train_dataset.classes

['O', 'R']

In [14]:
class DINOv2Classifier(nn.Module):
    def __init__(self, num_classes, model_checkpoint=CFG.model_checkpoint):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_checkpoint)
        hidden_size = self.backbone.config.hidden_size
        
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 1024), # 768 --> hidden_size
            nn.BatchNorm1d(1024),
            nn.GELU(),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        outputs = self.backbone(x)
        cls_token = outputs.last_hidden_state[:, 0]
        return self.head(cls_token)

In [15]:
class LightningModel(pl.LightningModule):
    def __init__(self, model, lr, num_classes):
        super().__init__()
        # saves hparams to load it later
        self.save_hyperparameters(ignore=['model']) 
        
        self.model = model
        self.lr = lr
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        # parameter groups: backbone (slow) & head (fast)
        param_groups = [
            {
                'params': self.model.backbone.parameters(),
                'lr': self.lr,
                'weight_decay': 1e-2
            },
            {
                'params': self.model.head.parameters(),
                'lr': self.lr * 10,
                'weight_decay': 1e-4
            }
        ]

        optimizer = torch.optim.AdamW(param_groups)
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.trainer.max_epochs
        )

        return [optimizer], [scheduler]
        
    def training_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.criterion(logits, labels)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.criterion(logits, labels)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == labels).float().mean()
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log('val_acc', acc, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def test_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.criterion(logits, labels)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == labels).float().mean()
        self.log('test_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log('test_acc', acc, prog_bar=True, on_step=True, on_epoch=True)
        return loss

In [16]:
# original model
dinov2 = DINOv2Classifier(num_classes=num_classes)

# lightning wrapper
model = LightningModel(dinov2, lr=1e-5, num_classes=num_classes)
model

LightningModel(
  (model): DINOv2Classifier(
    (backbone): Dinov2Model(
      (embeddings): Dinov2Embeddings(
        (patch_embeddings): Dinov2PatchEmbeddings(
          (projection): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
        )
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (encoder): Dinov2Encoder(
        (layer): ModuleList(
          (0-11): 12 x Dinov2Layer(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attention): Dinov2Attention(
              (attention): Dinov2SelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
              )
              (output): Dinov2SelfOutput(
                (dense): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.0, inplace=False)
     

In [17]:
trainer = Trainer(
    max_epochs=CFG.num_epoch,
    accelerator='gpu',
    num_sanity_val_steps=2, # set num_sanity_val_steps=0 to skip the initial validation check if needed
    precision='16-mixed'
)
trainer.fit(model, train_loader, val_loader)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ DINOv2Classifier │ 87.9 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 87.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 87.9 M                                                                                               
Total estimated model params size (MB): 351                                                                        
Modules in train mode: 11                                                                                          
Modules in eval mode: 224                                                                                          
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=8` reached.


Evaluate on test data

In [19]:
# create test dataset
test_dataset = ImageFolder(CFG.TEST_FOLDER_PATH)

# apply dataset transformation
test_dataset.transform = val_transform

# create dataloader
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [20]:
path_to_ckpt = trainer.checkpoint_callback.best_model_path

trained_model = LightningModel.load_from_checkpoint(
    checkpoint_path = path_to_ckpt,
    model = dinov2,
    num_classes = num_classes,
    lr=1e-5
)

trainer.test(trained_model, dataloaders=test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      test_acc_epoch       │    0.9649820923805237     │
│      test_loss_epoch      │    0.11867035180330276    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss_epoch': 0.11867035180330276,
  'test_acc_epoch': 0.9649820923805237}]